# AI Development Index — G7 + China (v2)

Composite AI development score using **5 indicators** that all have full G7+China coverage.

## Methodology
- **Indicator selection**: only datasets that contain data for all 8 countries (US, CN, UK, FR, DE, IT, JP, CA)
- **Excluded indicators** (data quality issues):
  - `Talent` — `fig_1.8.1` excludes China (LinkedIn-based, no China coverage)
  - `Adoption` — `fig_4.3.10` has narrow numeric range (16–44%) that distorts normalization
  - `Policy/bills` — `fig_8.4.3` is biased toward parliamentary legislation (against China's regulatory style)
- **Normalization**: `log10(x+1)` then **min-max scaling** to 0–100
  - Log compresses extreme US dominance
  - Min-max gives full 0–100 range per dimension

## Requirements
```bash
pip install pandas numpy matplotlib ipywidgets
```


## 1. Setup

In [ ]:
from pathlib import Path
import sys

repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / 'data').is_dir() and (p / 'src').is_dir()
)
src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import ipywidgets as widgets
from IPython.display import display

from common import find_repo_root

plt.rcParams['axes.unicode_minus'] = False

repo_root = find_repo_root(Path.cwd())
index_csv = repo_root / 'data' / 'number' / 'g7_china_datasets.csv'

assert index_csv.exists(), (
    f'{index_csv} not found. Run select_g7_china_data.ipynb first.'
)
print(f'Repo root:  {repo_root}')
print(f'Index file: {index_csv}')


## 2. Load the G7+China Dataset Index

In [ ]:
dataset_index = pd.read_csv(index_csv)
print(f'Loaded {len(dataset_index)} G7+China datasets')
dataset_index.head()


## 3. Country & Helper Definitions

In [ ]:
from common import COUNTRIES, COUNTRY_ALIASES as ALIASES, COUNTRY_COLORS as COLORS
from data_utils import extract_g7_china, load_figure

print('Helpers ready')


## 4. Define Indicators (5 Dimensions)

| Dimension | Figure | Notes |
|-----------|--------|-------|
| Infrastructure | `fig_1.3.2` | Number of data centers |
| R&D Output | `fig_1.1.3` | Cumulative notable ML models |
| Investment | `fig_4.2.12` | Total private AI investment ($B) |
| Industry | `fig_4.2.13` | Number of AI companies |
| Policy | `fig_8.3.5` | Sum of government AI initiatives across 11 sectors ($) |


In [ ]:
INDICATOR_CONFIG = {
    'Infrastructure': ('fig_1.3.2', 'Country',         'Number of Data Centers'),
    'R&D Output':     ('fig_1.1.3', 'Geographic area', 'Number of notable machine learning models'),
    'Investment':     ('fig_4.2.12','Country',         'Total Investment (in Billions of U.S. Dollars)'),
    'Industry':       ('fig_4.2.13','Country',         'Number of Companies'),
    # Policy: sum across all sector columns of fig_8.3.5
    'Policy':         ('fig_8.3.5', 'country',         '__sum_all_sectors__'),
}

missing = [name for name, (fig, _, _) in INDICATOR_CONFIG.items()
           if fig not in dataset_index['figure'].values]
assert not missing, f'Missing in G7+China index: {missing}'
print('All indicator figures verified.')


## 5. Load Raw Indicator Values

In [ ]:
raw = {}
for dim, (fig, ccol, vcol) in INDICATOR_CONFIG.items():
    df = load_figure(fig, dataset_index, repo_root)
    if vcol == '__sum_all_sectors__':
        sector_cols = [c for c in df.columns if c != ccol]
        df_num = df[sector_cols].apply(lambda s: pd.to_numeric(s, errors='coerce')).fillna(0)
        df = df.assign(_total=df_num.sum(axis=1))
        raw[dim] = extract_g7_china(df, ccol, '_total')
    else:
        raw[dim] = extract_g7_china(df, ccol, vcol)

raw_df = pd.DataFrame(raw)
print('Raw indicator values for G7 + China:')
display(raw_df.round(0))


## 6. Normalize: log10(x+1) → min-max → 0–100

The log step compresses the extreme dominance of large countries (US in particular).
The min-max step pins the lowest country to 0 and the highest to 100 per dimension.


In [ ]:
from ai_index import normalize_scores

norm_df = normalize_scores(raw_df)
print('Normalized scores (0 = lowest, 100 = highest in this group):')
display(norm_df)


## 7. Static Visualization (Equal Weights)

Default composite score using equal weights across the 5 dimensions.


In [ ]:
from ai_index import compute_weighted_scores

DIMENSIONS = list(norm_df.columns)
equal_weights = {dim: 1 for dim in DIMENSIONS}
default_scores, _ = compute_weighted_scores(norm_df, equal_weights)

fig, axes = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={'width_ratios': [1.1, 1]})

# Bar chart
colors = [COLORS[c] for c in default_scores.index]
axes[0].barh(default_scores.index[::-1], default_scores.values[::-1], color=colors[::-1])
axes[0].set_xlabel('Composite Score (0–100)')
axes[0].set_title('Ranking — Equal Weights', fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)
for i, v in enumerate(default_scores.values[::-1]):
    axes[0].text(v + 1, i, f'{v:.1f}', va='center', fontsize=9)
axes[0].set_xlim(0, 105)

# Radar (top 4)
ax2 = plt.subplot(122, projection='polar')
N = len(DIMENSIONS)
angles = [n / N * 2 * np.pi for n in range(N)] + [0]
top4 = default_scores.head(4).index.tolist()
for country in top4:
    vals = norm_df.loc[country].tolist()
    vals += vals[:1]
    ax2.plot(angles, vals, linewidth=2, label=country, color=COLORS[country])
    ax2.fill(angles, vals, alpha=0.1, color=COLORS[country])
ax2.set_xticks(angles[:-1])
ax2.set_xticklabels(DIMENSIONS, fontsize=9)
ax2.set_ylim(0, 100)
ax2.set_yticks([25, 50, 75, 100])
ax2.set_title('Profile — Top 4', fontweight='bold', pad=20)
ax2.legend(loc='upper right', bbox_to_anchor=(1.4, 1.0), fontsize=9)
ax2.grid(True)

plt.tight_layout()
plt.show()

default_scores.round(2)


## 8. Interactive GUI — Adjust Weights

Drag the sliders to set each dimension's weight. Weights are auto-normalized to sum to 100%.


In [ ]:
DEFAULT_WEIGHT = round(100 / len(DIMENSIONS), 1)

sliders = {
    dim: widgets.IntSlider(
        value=int(DEFAULT_WEIGHT),
        min=0, max=100, step=1,
        description=dim,
        style={'description_width': '120px'},
        layout=widgets.Layout(width='500px'),
        continuous_update=False,
    )
    for dim in DIMENSIONS
}

reset_btn = widgets.Button(description='Reset to Equal', button_style='info')
weight_display = widgets.HTML()


def update(**weights):
    scores, norm_w = compute_weighted_scores(norm_df, weights)

    rows = ''.join(
        f'<tr><td>{d}</td><td style="text-align:right">{w*100:.1f}%</td></tr>'
        for d, w in norm_w.items()
    )
    weight_display.value = f'''
    <table style="border-collapse:collapse">
      <tr><th style="text-align:left;padding-right:20px">Dimension</th>
          <th style="text-align:right">Weight</th></tr>
      {rows}
    </table>
    '''

    score_df = pd.DataFrame({'Composite Score': scores.round(2)})
    score_df.insert(0, 'Rank', range(1, len(score_df) + 1))
    print('Composite AI Development Score (G7 + China):')
    display(score_df)

    fig = plt.figure(figsize=(15, 6))
    gs = GridSpec(1, 2, width_ratios=[1.1, 1])

    ax1 = fig.add_subplot(gs[0])
    colors = [COLORS[c] for c in scores.index]
    ax1.barh(scores.index[::-1], scores.values[::-1], color=colors[::-1])
    ax1.set_xlabel('Composite Score (0–100)')
    ax1.set_title('Ranking', fontweight='bold')
    ax1.grid(axis='x', alpha=0.3)
    for i, v in enumerate(scores.values[::-1]):
        ax1.text(v + 1, i, f'{v:.1f}', va='center', fontsize=9)
    ax1.set_xlim(0, 105)

    ax2 = fig.add_subplot(gs[1], projection='polar')
    top4 = scores.head(4).index.tolist()
    N = len(DIMENSIONS)
    angles = [n / N * 2 * np.pi for n in range(N)] + [0]
    for country in top4:
        vals = norm_df.loc[country].tolist()
        vals += vals[:1]
        ax2.plot(angles, vals, linewidth=2, label=country, color=COLORS[country])
        ax2.fill(angles, vals, alpha=0.1, color=COLORS[country])
    ax2.set_xticks(angles[:-1])
    ax2.set_xticklabels(DIMENSIONS, fontsize=9)
    ax2.set_ylim(0, 100)
    ax2.set_yticks([25, 50, 75, 100])
    ax2.set_title('Profile of Top 4', fontweight='bold', pad=20)
    ax2.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=9)
    ax2.grid(True)

    plt.tight_layout()
    plt.show()


def on_reset(_):
    for s in sliders.values():
        s.value = int(DEFAULT_WEIGHT)


reset_btn.on_click(on_reset)

out = widgets.interactive_output(update, sliders)

ui = widgets.HBox([
    widgets.VBox([widgets.VBox(list(sliders.values())), reset_btn, weight_display]),
    out,
])
display(ui)


## 9. Export Current Result

In [ ]:
weights = {dim: sliders[dim].value for dim in DIMENSIONS}
scores, norm_w = compute_weighted_scores(norm_df, weights)

result = pd.DataFrame({
    'Country': scores.index,
    'Composite Score': scores.values.round(2),
    'Rank': range(1, len(scores) + 1),
})

out_path = repo_root / 'data' / 'number' / 'ai_dev_index_g7_china.csv'
result.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print('\nWeights used:')
for d, w in norm_w.items():
    print(f'  {d}: {w*100:.1f}%')
print()
display(result)


## 10. Browse Other G7+China Datasets

Edit `INDICATOR_CONFIG` (Section 4) to swap an indicator. Use this table to find candidates.


In [ ]:
dataset_index[['chapter', 'figure', 'country_count', 'rows', 'columns', 'column_names']]
